In [58]:
import numpy as np

import torch
import torch.nn as nn
import pandas as pd
import os
import importlib
import sys
from tqdm import tqdm
from torchvision import transforms
import torchvision.models as models 
from torch.utils.data import DataLoader

batch_size = 256

import torchsummary


torch.manual_seed(42)

In [2]:
# 1. Konfiguration
REPO_URL = "https://github.com/828w5tjjts-wq/Machine-Learning-Final-project.git"
PROJECT_DIR = "/kaggle/working/Machine-Learning-Final-project"

# 2. Prüfen, ob das Repo schon existiert
if os.path.exists(PROJECT_DIR):
    print("Repo existiert bereits. Ziehe neueste Änderungen (Pull)...")
    %cd {PROJECT_DIR}
    !git pull origin main
else:
    print("Repo existiert nicht. Klone von GitHub...")
    !git clone {REPO_URL} {PROJECT_DIR}
    %cd {PROJECT_DIR}

# 3. Pfad zum System-Pfad hinzufügen (damit Imports funktionieren)
sys.path.append(PROJECT_DIR)

print("Setup abgeschlossen. Projekt ist aktuell.")

Repo existiert nicht. Klone von GitHub...
Cloning into '/kaggle/working/Machine-Learning-Final-project'...
remote: Enumerating objects: 115, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 115 (delta 41), reused 76 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (115/115), 117.89 KiB | 3.10 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/kaggle/working/Machine-Learning-Final-project
Setup abgeschlossen. Projekt ist aktuell.


In [56]:
from src.models.architectures import MLPModel
from src.models.trainer import run_training
from src.data.loader import prepare_data
from src.data.loader import load_processed_data
from src.data.edit import split_data
from src.data.edit import standardize_data
from src.data.dataset import create_dataloaders
from src.models.tune import test_learning_rates
from src.models.trainer import train_model_with_early_stopping
from src.data.cleaner import clean_target
from src.data.dataset import create_dataloaders

In [14]:
# DO NOT CHANGE
use_cuda = True
use_cuda = False if not use_cuda else torch.cuda.is_available()
device = torch.device('cuda:0' if use_cuda else 'cpu')
torch.cuda.get_device_name(device) if use_cuda else 'cpu'
print('Using device', device)

Using device cpu


In [11]:
X_raw, y = load_processed_data('/kaggle/input/datasets/nilsmatthiessen/processed-data/Machine-Learning-Final-project/X_filtered.npy'
,'/kaggle/input/datasets/nilsmatthiessen/processed-data/Machine-Learning-Final-project/y.npy' )

X_train, X_val, X_test, y_train, y_val, y_test = split_data(X_raw, y)
X_train_scaled, X_val_scaled, X_test_scaled = standardize_data(X_train, X_val, X_test)
    
train_loader, val_loader, test_loader = create_dataloaders(
    X_train_scaled, X_val_scaled, X_test_scaled, 
    y_train, y_val, y_test, 
    batch_size=batch_size
    )
print("\nAlles bereit für das Training!")

sample_inputs, _ = next(iter(train_loader))
input_dim = sample_inputs.shape[1]

model_linear = MLPModel(input_dim, 1)
model_linear = model_linear.to(device)

loss_function = torch.nn.MSELoss() # Mean Squared Error für Regression
optimizer = torch.optim.Adam(model_linear.parameters(), lr=0.1) 
num_epochs = 100


Daten geladen: X Shape (9229, 78), y Shape (9229,)
Split: Train=6460, Val=1384, Test=1385
Scaling abgeschlossen.
DataLoader bereit (Batch Size 256):
  Train Batches: 26
  Val Batches: 6
  Test Batches: 6

Alles bereit für das Training!


In [49]:
model_mlp = MLPModel(input_dim=input_dim, output_dim=1).to(device)

# 2. Loss Function und Optimizer
loss_function = nn.MSELoss()
# Wir probieren 0.01, da MLPs empfindlicher auf große Sprünge reagieren können als lineare Modelle
optimizer = torch.optim.Adam(model_mlp.parameters(), lr=0.1) 


In [50]:
# Training starten
train_losses, val_losses, train_rmses, val_rmses = run_training(
    model=model_mlp,
    optimizer=optimizer,
    loss_function=loss_function,
    device=device,
    num_epochs=50,
    train_dataloader=train_loader,
    val_dataloader=val_loader
)

Starte Training für 50 Epochen...


<div><p>Epoch 1/50 | Train Loss (MSE): 9170.6347 | Train RMSE:       96.89€ | Val Loss (MSE):   8692.7747 | Val RMSE:         96.76€</p><p>Epoch 2/50 | Train Loss (MSE): 4256.8723 | Train RMSE:       65.41€ | Val Loss (MSE):   7073.7361 | Val RMSE:         87.88€</p><p>Epoch 3/50 | Train Loss (MSE): 3726.8818 | Train RMSE:       61.37€ | Val Loss (MSE):   6673.2107 | Val RMSE:         85.02€</p><p>Epoch 4/50 | Train Loss (MSE): 3972.7112 | Train RMSE:       63.14€ | Val Loss (MSE):   10259.3770 | Val RMSE:         105.16€</p><p>Epoch 5/50 | Train Loss (MSE): 3483.8970 | Train RMSE:       59.01€ | Val Loss (MSE):   4746.1900 | Val RMSE:         71.49€</p><p>Epoch 6/50 | Train Loss (MSE): 6040.1134 | Train RMSE:       78.54€ | Val Loss (MSE):   4089.9346 | Val RMSE:         66.16€</p><p>Epoch 7/50 | Train Loss (MSE): 4806.9262 | Train RMSE:       68.33€ | Val Loss (MSE):   5771.0637 | Val RMSE:         79.10€</p><p>Epoch 8/50 | Train Loss (MSE): 3672.8634 | Train RMSE:       60.99€ | Val Loss (MSE):   4596.2472 | Val RMSE:         70.57€</p><p>Epoch 9/50 | Train Loss (MSE): 4024.3204 | Train RMSE:       63.27€ | Val Loss (MSE):   6075.3668 | Val RMSE:         79.94€</p><p>Epoch 10/50 | Train Loss (MSE): 4470.3691 | Train RMSE:       66.85€ | Val Loss (MSE):   4269.3653 | Val RMSE:         68.21€</p><p>Epoch 11/50 | Train Loss (MSE): 2933.4521 | Train RMSE:       54.61€ | Val Loss (MSE):   9532.8468 | Val RMSE:         101.66€</p><p>Epoch 12/50 | Train Loss (MSE): 2190.6000 | Train RMSE:       47.20€ | Val Loss (MSE):   4972.9502 | Val RMSE:         73.82€</p><p>Epoch 13/50 | Train Loss (MSE): 2374.4908 | Train RMSE:       49.12€ | Val Loss (MSE):   2997.5451 | Val RMSE:         56.72€</p><p>Epoch 14/50 | Train Loss (MSE): 1966.7119 | Train RMSE:       44.56€ | Val Loss (MSE):   4134.7744 | Val RMSE:         67.01€</p><p>Epoch 15/50 | Train Loss (MSE): 5701.4430 | Train RMSE:       76.00€ | Val Loss (MSE):   7947.8274 | Val RMSE:         92.89€</p><p>Epoch 16/50 | Train Loss (MSE): 4638.9537 | Train RMSE:       68.60€ | Val Loss (MSE):   4422.3379 | Val RMSE:         68.87€</p><p>Epoch 17/50 | Train Loss (MSE): 2832.9723 | Train RMSE:       53.59€ | Val Loss (MSE):   4710.9456 | Val RMSE:         71.26€</p><p>Epoch 18/50 | Train Loss (MSE): 2958.5533 | Train RMSE:       53.37€ | Val Loss (MSE):   4999.3155 | Val RMSE:         74.02€</p><p>Epoch 19/50 | Train Loss (MSE): 3482.5958 | Train RMSE:       58.68€ | Val Loss (MSE):   8542.1888 | Val RMSE:         94.73€</p><p>Epoch 20/50 | Train Loss (MSE): 3905.0870 | Train RMSE:       62.98€ | Val Loss (MSE):   4812.8354 | Val RMSE:         72.53€</p><p>Epoch 21/50 | Train Loss (MSE): 2163.3732 | Train RMSE:       46.17€ | Val Loss (MSE):   4435.6597 | Val RMSE:         69.66€</p><p>Epoch 22/50 | Train Loss (MSE): 2002.2068 | Train RMSE:       44.76€ | Val Loss (MSE):   3654.4864 | Val RMSE:         63.03€</p><p>Epoch 23/50 | Train Loss (MSE): 2807.7774 | Train RMSE:       52.53€ | Val Loss (MSE):   3166.0217 | Val RMSE:         58.43€</p><p>Epoch 24/50 | Train Loss (MSE): 2725.8999 | Train RMSE:       52.59€ | Val Loss (MSE):   4239.7574 | Val RMSE:         67.82€</p><p>Epoch 25/50 | Train Loss (MSE): 2084.4986 | Train RMSE:       45.92€ | Val Loss (MSE):   3281.0133 | Val RMSE:         59.71€</p><p>Epoch 26/50 | Train Loss (MSE): 2120.2129 | Train RMSE:       46.35€ | Val Loss (MSE):   3515.2674 | Val RMSE:         61.84€</p><p>Epoch 27/50 | Train Loss (MSE): 2129.7443 | Train RMSE:       46.26€ | Val Loss (MSE):   3271.1295 | Val RMSE:         59.89€</p><p>Epoch 28/50 | Train Loss (MSE): 1791.5611 | Train RMSE:       42.78€ | Val Loss (MSE):   3308.5119 | Val RMSE:         60.07€</p><p>Epoch 29/50 | Train Loss (MSE): 2242.6902 | Train RMSE:       47.81€ | Val Loss (MSE):   3960.9348 | Val RMSE:         65.83€</p><p>Epoch 30/50 | Train Loss (MSE): 1747.8342 | Train RMSE:       41.27€ | Val Loss (MSE):   3747.2326 | Val RMSE:         64.02€</p><p>Epoch 31/50 | Train Loss (MSE): 3080.3

Training beendet.


In [ ]:
# 1. Den Index des minimalen Validation RMSE finden
best_epoch_idx = np.argmin(val_rmses)

# 2. Die Werte an diesem Index auslesen
best_epoch = best_epoch_idx + 1 # Epochen fangen bei 1 an für Menschen
best_val_rmse = val_rmses[best_epoch_idx]
best_train_rmse = train_rmses[best_epoch_idx]
best_val_loss = val_losses[best_epoch_idx]

print("="*50)
print(f"BESTER DURCHLAUF (Epoche {best_epoch}):")
print("-" * 50)
print(f"Train RMSE: {best_train_rmse:.2f}€")
print(f"Val RMSE:   {best_val_rmse:.2f}€  <-- BESTES ERGEBNIS")
print(f"Val Loss:   {best_val_loss:.4f}")
print("="*50)

In [13]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [54]:
resnet = models.resnet18(pretrained=True)
resnet = nn.Sequential(*list(resnet.children())[:-1]) 
resnet.to(device)
resnet.eval() # Evaluation mode (kein Training nötig)


Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Con

In [55]:
def extract_features_from_urls(url_list):
    features_list = []
    for url in tqdm(url_list, desc="Extrahiere Bild-Features"):
        try:
            response = requests.get(url, timeout=5)
            img = Image.open(BytesIO(response.content)).convert('RGB')
            img_tensor = preprocess(img).unsqueeze(0).to(device)
            
            with torch.no_grad():
                features = resnet(img_tensor)
            
            features_list.append(features.flatten().cpu().numpy())
        except:
            # Falls Fehler: Null-Vektor
            features_list.append(np.zeros(512))
            
    return np.array(features_list)


In [26]:
print("Lade listings.csv...")
df = pd.read_csv("/kaggle/input/datasets/nilsmatthiessen/listings/listings-2.csv")
df = clean_target(df)

Lade listings.csv...


In [ ]:
print("Lade Bilder")
df = pd.read_csv("/kaggle/input/datasets/nilsmatthiessen/listings/listings-2.csv")
df = clean_target(df)
urls = df['picture_url'].tolist()
print("Starte Bild-Download und Feature Extraktion...")
X_images = extract_features_from_urls(urls)

# Speichern (damit wir das nicht neu machen müssen)
np.save('X_images.npy', X_images)
print(f"Bild-Features gespeichert! Shape: {X_images.shape}")


try :
    print(f"\nLade existierende Bild-Features von '{image_path}'...")
    X_images = np.load('X_images.npy', allow_pickle=True)
else:
    print("Lade Bilder")
    df = pd.read_csv("/kaggle/input/datasets/nilsmatthiessen/listings/listings-2.csv")
    df = clean_target(df)
    urls = df['picture_url'].tolist()
    print("Starte Bild-Download und Feature Extraktion...")
    X_images = extract_features_from_urls(urls)

    np.save('X_images.npy', X_images)
    print(f"Bild-Features gespeichert! Shape: {X_images.shape}")


In [ ]:
num_samples = len(df)

train_split = int(0.7 * num_samples)
val_split = int(0.85 * num_samples)

train_idx = indices[:train_split]
val_idx = indices[train_split:val_split]
test_idx = indices[val_split:]

# Indizes auf beide Datensätze anwenden
X_train_img, X_val_img, X_test_img = X_img[train_idx], X_img[val_idx], X_img[test_idx]

print(f"Split fertig: Train={len(train_idx)}, Val={len(val_idx)}, Test={len(test_idx)}")

# --- 3. SCALING ---
print("\nStarte Scaling...")

# A. Tabellarische Daten skalieren (für späteres Multi-Modal Training)
mean_tab = X_train_tab.mean(axis=0)
std_tab = X_train_tab.std(axis=0)
std_tab[std_tab == 0] = 1.0
X_train_tab_s = (X_train_tab - mean_tab) / std_tab
X_val_tab_s = (X_val_tab - mean_tab) / std_tab
X_test_tab_s = (X_test_tab - mean_tab) / std_tab

# B. Bilddaten skalieren (für das Nur-Bilder Training HIER)
mean_img = X_train_img.mean(axis=0)
std_img = X_train_img.std(axis=0)
std_img[std_img == 0] = 1.0
X_train_img_s = (X_train_img - mean_img) / std_img
X_val_img_s = (X_val_img - mean_img) / std_img
X_test_img_s = (X_test_img - mean_img) / std_img

print("Scaling fertig.")

# --- 4. DATALOADER ERSTELLEN (Nur für Bilder) ---
print("\nErstelle DataLoader für Bild-Modell...")

train_loader_img, val_loader_img, test_loader_img = create_dataloaders(
    X_train=X_train_img_s,
    X_val=X_val_img_s,
    X_test=X_test_img_s,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
    batch_size=64
)

print("\n✅ ALLES BEREIT!")
print(f"Train Loader Batches: {len(train_loader_img)}")
print(f"Val Loader Batches:   {len(val_loader_img)}")

In [47]:
sample_inputs, _ = next(iter(train_loader))
input_dim = sample_inputs.shape[1]
print(f"Input Dimension: {input_dim}")

model_MLP = MLPModel(input_dim=input_dim, output_dim=1).to(device)
optimizer = torch.optim.Adam(model_MLP.parameters(), lr=0.1)
loss_function = nn.MSELoss()

Input Dimension: 78


In [46]:
# ... (Modell, Optimizer, etc. sind definiert) ...

# Training starten
train_losses, val_losses, train_rmses, val_rmses = run_training(
    model=model_MLP,
    optimizer=optimizer,
    loss_function=loss_function,
    device=device,
    num_epochs=50,
    train_dataloader=train_loader,
    val_dataloader=val_loader
)

# Besten Durchlauf finden
best_epoch_idx = np.argmin(val_rmses)
print(f"\nBester Durchlauf war Epoche {best_epoch_idx+1} mit Val RMSE: {val_rmses[best_epoch_idx]:.2f}€")

Starte Training für 50 Epochen...


<div><p>Epoch 1/50 | Train Loss (MSE): 9486.8942 | Train RMSE:       97.71€ | Val Loss (MSE):   11707.7840 | Val RMSE:         112.43€</p><p>Epoch 2/50 | Train Loss (MSE): 4342.9666 | Train RMSE:       66.41€ | Val Loss (MSE):   7943.0193 | Val RMSE:         92.42€</p><p>Epoch 3/50 | Train Loss (MSE): 3898.4008 | Train RMSE:       62.55€ | Val Loss (MSE):   12404.3387 | Val RMSE:         115.85€</p><p>Epoch 4/50 | Train Loss (MSE): 3280.8033 | Train RMSE:       57.16€ | Val Loss (MSE):   12808.2840 | Val RMSE:         117.98€</p><p>Epoch 5/50 | Train Loss (MSE): 4247.5228 | Train RMSE:       65.56€ | Val Loss (MSE):   5953.4257 | Val RMSE:         79.86€</p><p>Epoch 6/50 | Train Loss (MSE): 4538.5709 | Train RMSE:       68.15€ | Val Loss (MSE):   6073.6046 | Val RMSE:         80.72€</p><p>Epoch 7/50 | Train Loss (MSE): 4436.7731 | Train RMSE:       66.22€ | Val Loss (MSE):   7404.6509 | Val RMSE:         89.70€</p><p>Epoch 8/50 | Train Loss (MSE): 6202.0050 | Train RMSE:       79.60€ | Val Loss (MSE):   4561.1377 | Val RMSE:         70.36€</p><p>Epoch 9/50 | Train Loss (MSE): 3440.9302 | Train RMSE:       58.22€ | Val Loss (MSE):   4347.5647 | Val RMSE:         68.30€</p><p>Epoch 10/50 | Train Loss (MSE): 3015.6469 | Train RMSE:       55.10€ | Val Loss (MSE):   6555.6746 | Val RMSE:         84.15€</p><p>Epoch 11/50 | Train Loss (MSE): 3432.6399 | Train RMSE:       58.26€ | Val Loss (MSE):   6950.0742 | Val RMSE:         85.73€</p><p>Epoch 12/50 | Train Loss (MSE): 4174.4440 | Train RMSE:       65.17€ | Val Loss (MSE):   3459.1552 | Val RMSE:         60.66€</p><p>Epoch 13/50 | Train Loss (MSE): 3079.8038 | Train RMSE:       55.90€ | Val Loss (MSE):   4331.5482 | Val RMSE:         68.30€</p><p>Epoch 14/50 | Train Loss (MSE): 3467.8437 | Train RMSE:       59.34€ | Val Loss (MSE):   3555.1029 | Val RMSE:         61.87€</p><p>Epoch 15/50 | Train Loss (MSE): 3147.4042 | Train RMSE:       56.54€ | Val Loss (MSE):   3895.7344 | Val RMSE:         64.82€</p><p>Epoch 16/50 | Train Loss (MSE): 4029.8727 | Train RMSE:       63.11€ | Val Loss (MSE):   5853.0042 | Val RMSE:         78.29€</p><p>Epoch 17/50 | Train Loss (MSE): 4995.6422 | Train RMSE:       71.00€ | Val Loss (MSE):   4752.3006 | Val RMSE:         71.71€</p><p>Epoch 18/50 | Train Loss (MSE): 2771.0471 | Train RMSE:       52.86€ | Val Loss (MSE):   3784.9305 | Val RMSE:         63.39€</p><p>Epoch 19/50 | Train Loss (MSE): 2746.6936 | Train RMSE:       52.64€ | Val Loss (MSE):   2930.9218 | Val RMSE:         55.99€</p><p>Epoch 20/50 | Train Loss (MSE): 2429.0191 | Train RMSE:       49.55€ | Val Loss (MSE):   3296.2457 | Val RMSE:         59.67€</p><p>Epoch 21/50 | Train Loss (MSE): 2318.7662 | Train RMSE:       48.60€ | Val Loss (MSE):   3678.1124 | Val RMSE:         63.27€</p><p>Epoch 22/50 | Train Loss (MSE): 3427.6846 | Train RMSE:       59.12€ | Val Loss (MSE):   3110.9945 | Val RMSE:         58.12€</p><p>Epoch 23/50 | Train Loss (MSE): 2276.4719 | Train RMSE:       47.82€ | Val Loss (MSE):   3324.2593 | Val RMSE:         60.19€</p><p>Epoch 24/50 | Train Loss (MSE): 2545.0462 | Train RMSE:       46.83€ | Val Loss (MSE):   6103.0831 | Val RMSE:         80.14€</p><p>Epoch 25/50 | Train Loss (MSE): 3961.9036 | Train RMSE:       63.45€ | Val Loss (MSE):   4121.0285 | Val RMSE:         66.14€</p><p>Epoch 26/50 | Train Loss (MSE): 2903.7183 | Train RMSE:       53.85€ | Val Loss (MSE):   2101.1569 | Val RMSE:         47.41€</p><p>Epoch 27/50 | Train Loss (MSE): 3109.2540 | Train RMSE:       55.77€ | Val Loss (MSE):   3647.9180 | Val RMSE:         62.46€</p><p>Epoch 28/50 | Train Loss (MSE): 2542.1623 | Train RMSE:       50.58€ | Val Loss (MSE):   2658.3872 | Val RMSE:         53.60€</p><p>Epoch 29/50 | Train Loss (MSE): 2198.9184 | Train RMSE:       46.44€ | Val Loss (MSE):   2972.8167 | Val RMSE:         56.40€</p><p>Epoch 30/50 | Train Loss (MSE): 2026.9119 | Train RMSE:       44.11€ | Val Loss (MSE):   3369.6050 | Val RMSE:         60.52€</p><p>Epoch 31/50 | Train Loss (MSE): 209

Training beendet.

Bester Durchlauf war Epoche 26 mit Val RMSE: 47.41€
